<a href="https://colab.research.google.com/github/emmanuelle-bul/AI_Exercises/blob/main/OpenAI_OpenRouter_ResponsesWithHistory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field  ## за дефиниране на class
from google.colab import userdata

openai_api_key = userdata.get('OPENAI_API_KEY') #взимаме по име Secret (API key), с който ще достъпваме сървъра на OpenAI
openai_client = OpenAI(api_key=openai_api_key) #създаваме клиент

# със същото SDK можем да ползваме клиент към OpenRouter
openrouter_api_key = userdata.get('OPENROUTER_API_KEY') #взимаме по име Secret (API key), с който ще достъпваме сървъра на OpenRouter
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url='https://openrouter.ai/api/v1') #създаваме клиент


In [ ]:
openai_models = openai_client.models.list()

In [ ]:
openai_models.data
# Когато експериментираме да използваме по-евтини модели -> ...nano, ...mini

[Model(id='text-embedding-ada-002', created=1671217299, object='model', owned_by='openai-internal', shutdown_date=None),
 Model(id='whisper-1', created=1677532384, object='model', owned_by='openai-internal', shutdown_date='2027-02-26'),
 Model(id='gpt-3.5-turbo', created=1677610602, object='model', owned_by='openai', shutdown_date='2026-10-23'),
 Model(id='tts-1', created=1681940951, object='model', owned_by='openai-internal', shutdown_date=None),
 Model(id='gpt-3.5-turbo-16k', created=1683758102, object='model', owned_by='openai-internal', shutdown_date=None),
 Model(id='gpt-4-0613', created=1686588896, object='model', owned_by='openai', shutdown_date='2026-10-23'),
 Model(id='gpt-4', created=1687882411, object='model', owned_by='openai', shutdown_date='2026-10-23'),
 Model(id='davinci-002', created=1692634301, object='model', owned_by='system', shutdown_date='2026-09-28'),
 Model(id='babbage-002', created=1692634615, object='model', owned_by='system', shutdown_date='2026-09-28'),
 Mo

In [ ]:
hello_response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input="Hello!"
)

In [ ]:
hello_response

Response(id='resp_0bb82003f800a78c006ab3c4dd17cc87d081679a9b88496543', created_at=1790166237.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0bb82003f800a78c006ab3c4dd632087d099785edf1ca77769', content=[ResponseOutputText(annotations=[], text='Hello! 👋 How can I help you today?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1790166237.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention='24h', reasoning=Reasoning(context='current_turn', effort='none', generate_summary=None, mode='standard', summary=None), safety_identifier=None, service_tier='default',

In [ ]:
introduction_response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input="Hello! My name is Emi. What is your name? Help me plan my day."
)

In [ ]:
introduction_response

Response(id='resp_024c0a5f28e3f317006ab3c68e531087d0b245225f30f17f61', created_at=1790166670.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_024c0a5f28e3f317006ab3c68f87c887d0b716c164001e9c74', content=[ResponseOutputText(annotations=[], text='Hi Emi! Nice to meet you 😊  \nI’m ChatGPT—your AI assistant.\n\nI can definitely help you plan your day. To set it up well, tell me:\n\n1) What time is it for you right now?  \n2) What time do you want to be done / go to bed?  \n3) Any fixed commitments today? (work, school, appointments)  \n4) What are your top 3 priorities?  \n5) How much time do you want for breaks or fun?\n\nReply with whatever you know, and I’ll turn it into a clear, realistic schedule.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', t

In [ ]:
aquintance_response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input="Hello! Do you know my name?"
)

In [ ]:
aquintance_response # въпреки, че сме се предствили по-горе -> той все едно започва нова сесия. Когато се обърнем към responses.create API означава, че все едно започваме нова сесия. OpenAi не знае за предишните обръщения.

Response(id='resp_0e262cc106f1552b006ab3c71ad91087d093ae640089dbc512', created_at=1790166810.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0e262cc106f1552b006ab3c71ca35087d0b33b2f832e243f04', content=[ResponseOutputText(annotations=[], text='Hello! I don’t know your name from this chat. If you tell me your name (or what you’d like me to call you), I’ll use it.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1790166812.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention='24h', reasoning=Reasoning(context='current_turn', effort='none', generate_summary

In [ ]:
# Обвързване на отговори в една сесия

# има различни роли в input параметъра ->
# роля "developer" = системния промпт (в него е описано каква е неговата роля, как трябва да се държи агента, инструкции);
# роля "user" = това е конкретен промпт от потребителя, върху който агента трябва да размишлява;
# роля "assistant" = когато връща отговор го връща от ролята на асистент
aquintance_response2 = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=[
        {
            "role": "user",
            "content": "Hello! My name is Emi. What is your name? Help me plan my day."
        },
        *introduction_response.output,                   #елементите на отговора ще бъдат поставени
        {
             "role": "user",
            "content": "Hello! Do you know my name?"
        }
    ]
)

In [ ]:
aquintance_response2

Response(id='resp_01645d91c392a0fb006ab3cb47415087d092f5699fb9c1371c', created_at=1790167879.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_01645d91c392a0fb006ab3cb486d1487d08fe627afa9c86355', content=[ResponseOutputText(annotations=[], text='Hi Emi! Yes—I know your name is **Emi** because you told me earlier.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1790167880.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention='24h', reasoning=Reasoning(context='current_turn', effort='none', generate_summary=None, mode='standard', summary=None), safety_identi

In [ ]:
# instructions - инструкции за агента; трябва да се добавят към всеки request;

aquintance_response_instructions = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=[
        {
            "role" : "developer", # или "system"
            "content" : "You are an expert in everyday engagments planning..." # описва се основната роля на този агент, възможности, преференции за стил output
        },
        {
            "role": "user",
            "content": "Hello! My name is Emi. What is your name? Help me plan my day."
        },
        *introduction_response.output,                   #елементите на отговора ще бъдат поставени
        {
             "role": "user",
            "content": "Hello! Do you know my name?"
        }
    ]
)

In [ ]:
# previous_response_id - САМО ЗА OpenAI
aquintance_response_previous_response_id = openai_client.responses.create(
    model="gpt-5.4-nano",
    input="Hello! Do you know my name?",
    previous_response_id=introduction_response.id


)

In [ ]:
aquintance_response_previous_response_id


Response(id='resp_08216887ce20fe5d006ab3e6faa43c87d0abad3e4d5b7ac394', created_at=1790174970.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_08216887ce20fe5d006ab3e6fb16d087d0b67e6d43e1fe46c4', content=[ResponseOutputText(annotations=[], text='Hi Emi! Yes—I can see your name from your messages: you said “My name is Emi.”', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1790174971.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id='resp_08216887ce20fe5d006ab3e6f21b9887d0b04f942eaba8e210', prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention='24h', reasoning=Reasoning(context='current_turn', effort='none', gener

In [ ]:
# Conversations - остават в сървърите докато не ги изтрием, докато requests изчистват се автоматично на определено време от сървърите на OpenAI
# можем да ги видим в https://platform.openai.com/logs?api=conversations
openai_conversation = openai_client.conversations.create()


In [ ]:
openai_conversation

Conversation(id='conv_6ab3e9923cac8195aa28d901bdfeac5905e039c740df0983', created_at=1790175634, metadata={}, object='conversation')

In [ ]:
gretings_response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input="Hello! Who are you?",
    conversation=openai_conversation.id
)

In [ ]:
gretings_response

Response(id='resp_06f62731faf25c78006ab3eb83c98881959ca370db17df5cc9', created_at=1790176132.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_06f62731faf25c78006ab3eb848c408195b26553b3a2830995', content=[ResponseOutputText(annotations=[], text='Hi! I’m an AI assistant from OpenAI. I can help answer questions, explain things, write and edit text, and assist with tasks like coding or planning.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1790176132.0, conversation=Conversation(id='conv_6ab3eb07d1c88195aed98d72e520846006f62731faf25c78'), max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_c

In [ ]:
next_response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input="Thank you!",
    conversation=openai_conversation.id
)

In [ ]:
## Structured output - JSON

class Test(BaseModel):
  """A test class for structured output functionalities"""      # така се документира клас
  key: str = Field(description= "Key value of this test object")
  value: str

In [ ]:
Test.model_json_schema()

{'description': 'A test class for structured output functionalities',
 'properties': {'key': {'description': 'Key value of this test object',
   'title': 'Key',
   'type': 'string'},
  'value': {'title': 'Value', 'type': 'string'}},
 'required': ['key', 'value'],
 'title': 'Test',
 'type': 'object'}

In [ ]:
class CalendarEvent(BaseModel):
  name: str
  short_description: str
  duration_h: float = Field(description="Duration of the event in hours.")

In [ ]:
# отново се обръща към responses.create endpoint, но ще включва този parsing + validations => ParsedResponse; Ако използваме create -> отговорът (Response) ще е отново JSON, но като текст, който трябва сами да си парсваме

structured_output = openai_client.responses.parse(
    model="gpt-5.4-nano",
    input= "Generate a ramdom calender event for thursday morning.",
    text_format= CalendarEvent
)

In [ ]:
structured_output.output_parsed.short_description

'Quick Thursday morning check-in to align on priorities for the week and confirm next steps.'

In [ ]:
## Streaming
# резултатът е ResponseStreamManager

with openai_client.responses.stream(model="gpt-5.4-nano", input= "Describe all popular types of pizza with their ingredients and rate them by best flavour.") as basic_stream:
  for event in basic_stream:    # така обхождаме стрийма
    if 'delta' in event.dict():
      print(event.delta, end = ' ', flush= True)  # flush - форсира изпълнението на print, да го видим веднага на екрана

Below  are  many  of  the  most  popular  pizza  styles ,  with  typical  ingredients  and  a  ** relative  “ fl avour 

/tmp/ipykernel_1301/4025785599.py:6: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  if 'delta' in event.dict():


 rating ” **  based  on  the  * average  consensus *  of  taste  ( not  just  “ sp ici est ”  or  “ most  expensive ”).  Ratings  are  ** subject ive **  ( people  vary ),  but  the  order  usually  matches  what  many  pizza  lovers  consider  most  satisfying .

 ##  Top  Tier  ( usually  best  flavour )

 ###   1 )  Ne ap olitan  ( Pizza  N apo let ana )  —  ** 9 . 7 / 10 **
 ** Typical  ingredients :**   
 -  San  Mar zano  tomatoes  ( or  similar ),  mozzarella  di  buf ala  or  f ior  di  latte   
 -  Fresh  basil ,  extra - vir gin  olive  oil   
 -  Salt ,  yeast ,  flour ,  water  ( thin ,  airy  dough )   
 ** Popular  variants :**  Mar gher ita ,  Marin ara  ( no  mozzarella ),  Marin ara  is  also  excellent .

 ---

 ###   2 )  New  York – Style  Cheese  Pizza  —  ** 9 . 4 / 10 **
 ** Typical  ingredients :**   
 -  Tomato  sauce  ( season ed ),  mozzarella  ( often  whole -m ilk ),  oregano / gar lic   
 -  Dough  with  a  more  robust  chew   
 -  Sometimes  olive  oil  

/usr/local/lib/python3.13/dist-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `ParsedResponseOutputMessage[TypeVar]` - serialized value may not be as expected [field_name='output', input_value=ParsedResponseOutputMessa...', phase='final_answer'), input_type=ParsedResponseOutputMessage[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `ParsedResponseFunctionToolCall` - serialized value may not be as expected [field_name='output', input_value=ParsedResponseOutputMessa...', phase='final_answer'), input_type=ParsedResponseOutputMessage[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `ResponseFileSearchToolCall` - serialized value may not be as expected [field_name='output', input_value=ParsedResponseOutputMessa...', phase='final_answer'), input_type=ParsedResponseOutputMessage[TypeVar]])
  PydanticSerializationUnexpectedValue(Expected `ResponseFunctionWebSearch` - serialized value may not be as e

In [ ]:
## OPENROUTER

openrouter_models = openai_client.models.list()

In [ ]:
openrouter_models.data

[Model(id='o1', created=1734375816, object='model', owned_by='system', shutdown_date='2026-10-23'),
 Model(id='gpt-5.4-mini', created=1773451123, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-4-turbo-2024-04-09', created=1712601677, object='model', owned_by='system', shutdown_date='2026-10-23'),
 Model(id='gpt-3.5-turbo-instruct', created=1692901427, object='model', owned_by='system', shutdown_date='2026-09-28'),
 Model(id='gpt-5-nano', created=1754426384, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-5.6-sol', created=1782228018, object='model', owned_by='system', shutdown_date=None),
 Model(id='babbage-002', created=1692634615, object='model', owned_by='system', shutdown_date='2026-09-28'),
 Model(id='gpt-image-2.5-flare', created=1788563147, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-5-pro', created=1759469822, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-5.4-2026-03-05', created=

In [ ]:
hello_response_openrouter = openrouter_client.responses.create(
    model="ibm-granite/granite-4.2-8b",
    input="Hello!"
)


In [ ]:
hello_response_openrouter

Response(id='gen-1790254910-bRRnJG8qfZFGW4WeKqpI', created_at=1790254910.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='ibm-granite/granite-4.2-8b', object='response', output=[ResponseReasoningItem(id='rs_tmp_wtohhvl9n4', summary=[], type='reasoning', content=[Content(text='Okay, the user said "Hello!" so I should respond politely. Keep it friendly and open-ended to encourage them to ask their question.\n', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'), ResponseOutputMessage(id='msg_tmp_2exesskwyg3', content=[ResponseOutputText(annotations=[], text="Hello! 😊 How can I help you today? Feel free to ask me any question—whether it's about explaining something, solving a problem, brainstorming ideas, or just chatting! 🌟", type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=Fa

In [ ]:
hello_response_openrouter.output_text

"Hello! 😊 How can I help you today? Feel free to ask me any question—whether it's about explaining something, solving a problem, brainstorming ideas, or just chatting! 🌟"

In [ ]:
hello_response_openrouter.model

'ibm-granite/granite-4.2-8b'